<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_experiment_feature-aggregator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [153]:
import os, glob, json
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

In [154]:
PATH = '/content/Cross-Platform-Binary-Code-Similarity-Detection'

# Wandb Login

In [155]:
import wandb
wandb.login()

True

# Arch-Opt Parsing Function

In [156]:
_ARCHES = ('x86', 'arm32', 'arm', 'mips32', 'mips')
_OPTS   = ('O0', 'O1', 'O2', 'O3')

In [157]:
def parse_arch_opt(src):
  arch = next((a for a in _ARCHES if f'-{a}-' in src), None)
  opt  = next((o for o in _OPTS   if f'-{o}' in src), None)
  return arch, opt

# Load Dataset

In [158]:
def load_dataset(dirs, min_blocks=4, n_feat=8):
  if isinstance(dirs, str):
    dirs = [dirs]

  files = []
  for d in dirs:
    files += glob.glob(os.path.join(d, '*.json'))

  if not files:
    raise FileNotFoundError(f'no .json under {dirs}')

  funcs, fnames, dropped = [], [], 0
  for fp in files:
    with open(fp) as fh:
      for line in fh:
        line = line.strip()
        if not line: continue

        r = json.loads(line)
        feats = r['features']
        if len(feats) < min_blocks:
          dropped += 1
          continue

        r['X'] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
        r['succs'] = r['succs'] if 'succs' in r else None
        r['arch'], r['opt'] = parse_arch_opt(r.get('src', ''))

        funcs.append(r)
        fnames.append(r['fname'])

  uniq = sorted(set(fnames))
  classes = {n: i for i, n in enumerate(uniq)}
  labels = np.array([classes[n] for n in fnames], dtype=np.int64)
  print(f'[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}')

  return funcs, labels, classes


# Split Dataset

In [159]:
def make_split(labels, ratios=(.8, .1, .1), seed=1337):
  n_cls = labels.max() + 1
  rng = np.random.default_rng(seed)
  cls_perm = rng.permutation(n_cls)

  n_train = int(ratios[0] * n_cls)
  n_val   = int(ratios[1] * n_cls)

  train_cls = set(cls_perm[:n_train].tolist())
  val_cls   = set(cls_perm[n_train: n_train + n_val].tolist())
  test_cls  = set(cls_perm[n_train + n_val:].tolist())

  train_mask = np.array([c in train_cls for c in labels])
  val_mask   = np.array([c in val_cls   for c in labels])
  test_mask  = np.array([c in test_cls  for c in labels])

  print(f'[split] train={train_mask.sum()} val={val_mask.sum()} test={test_mask.sum()}')
  return train_mask, val_mask, test_mask

# Functions for Metrics

In [160]:
def _normalize(V, eps=1e-10):
  return V / (np.linalg.norm(V, axis=1, keepdims=True) + eps)

In [161]:
def _group_by_class(labels):
  d = {}
  for i, c in enumerate(labels):
    d.setdefault(int(c), []).append(i)
  return d

In [162]:
# axis query and target belong to
def _axis(qa, qo, ta, to):
  if qa == ta and qo != to: return 'XO'
  if qa != ta and qo == to: return 'XA'
  if qa != ta and qo != to: return 'XM'
  return 'same'

# Metrics

### Auc Metrics

In [163]:
def auc_pairs(E, score_fn, labels, n_pairs=8000, seed=1337):
  rng = np.random.default_rng(seed);
  by = _group_by_class(labels)

  # leave functions which have at least two compiled versions (eligible for positive pairs)
  pos_c = [c for c, v in by.items() if len(v) >= 2]

  s, y = [], []

  # same functions with different arch-opt pair
  for _ in range(n_pairs // 2):
    c = pos_c[rng.integers(len(pos_c))]
    i, j = rng.choice(by[c], 2, replace=False)
    s.append(float(score_fn(E, i, j)))
    y.append(1)

  # different functions
  n = len(labels); got = 0
  while got < n_pairs // 2:
    i, j = rng.integers(n), rng.integers(n)
    if labels[i] == labels[j]:
      continue
    s.append(float(score_fn(E, i, j)))
    y.append(0)
    got += 1

  return s, y

### Retrieval Metrics

In [164]:
def retrieval(E, score_fn, labels, arches, opts, pool_sizes=(10, 100, 1000), n_queries=1000, ks=(1, 5, 10), seed=1337, per_axis=True):
  rng = np.random.default_rng(seed)
  by = _group_by_class(labels)

  pos_c = [c for c, v in by.items() if len(v) >= 2]
  axes = ['XO', 'XA', 'XM'] if per_axis else []
  out = {}

  for pool in pool_sizes:
    if pool > len(labels): continue

    # init recall hit for each top k
    rec = {k: 0 for k in ks}

    # init running sum of 1/rank
    mrr = 0.0

    # init number of queries processed
    Q = 0

    # init same metrics but for each axis now
    arec = {ax: {k: 0 for k in ks} for ax in axes}
    amrr = {ax: 0.0 for ax in axes}
    acnt = {ax: 0 for ax in axes}


    for _ in range(n_queries):
      # choose class and positive pairs of functions
      c = pos_c[rng.integers(len(pos_c))]
      q, tgt = rng.choice(by[c], 2, replace=False)

      # choosing functions with distinct classes
      dist = []
      while len(dist) < pool - 1:
        d = rng.integers(len(labels))
        if labels[d] != c:
          dist.append(d)

      cand = np.array([tgt] + dist)
      sims = score_fn(E, q, cand)

      # calculate where positive pair is ranked
      rank = np.where(np.argsort(-sims) == 0)[0][0] + 1

      # update metrics
      mrr += 1.0 / rank
      for k in ks:
        if rank <= k: rec[k] += 1
      Q += 1

      # if per_axis is enabled, log same metrics but separately for each (q, tgt) axis
      if per_axis:
        ax = _axis(arches[q], opts[q], arches[tgt], opts[tgt])
        if ax in arec:
          amrr[ax] += 1.0 / rank; acnt[ax] += 1
          for k in ks:
            if rank <= k: arec[ax][k] += 1

    # save results to out
    for k in ks:
      out[f'recall@{k}_pool{pool}'] = rec[k] / Q
    out[f'mrr_pool{pool}'] = mrr / Q

    for ax in axes:
      if acnt[ax] == 0:
        continue
      for k in ks:
        out[f'{ax}_recall@{k}_pool{pool}'] = arec[ax][k] / acnt[ax]
      out[f'{ax}_mrr_pool{pool}'] = amrr[ax] / acnt[ax]

  return out

# Evaluate

In [165]:
def evaluate(embedder, score_fn, funcs, labels, pool_sizes=(10, 100, 1000), n_pairs=8000, n_queries=500, ks=(1, 5, 10), per_axis=True):
  E = embedder.embed_batch(funcs)
  arches = [f.get('arch') for f in funcs]
  opts   = [f.get('opt')  for f in funcs]

  # roc and pr
  s, y = auc_pairs(E, score_fn, labels, n_pairs=n_pairs)
  roc_auc, pr_auc = roc_auc_score(y, s), average_precision_score(y, s)

  # retrieval
  retr = retrieval(E, score_fn, labels, arches, opts, pool_sizes=pool_sizes, n_queries=n_queries, ks=ks, per_axis=per_axis)

  return  {'roc_auc': roc_auc, 'pr_auc': pr_auc, 's': s, 'y': y, **retr}

# Printing & Plotting & Logging Data

### Plotting

In [166]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve

AXIS_LABELS = {'XO': 'XO cross-opt', 'XA': 'XA cross-arch', 'XM': 'XM cross-both'}

def _pools_in(r, pools):
    return [p for p in pools if f'recall@1_pool{p}' in r]

def make_plots(results, axes, pools, ks, model_name='model'):
    figs = {}
    tags = list(results.keys())
    C = plt.cm.tab10(np.linspace(0, 1, 10))

    # ROC curves
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            fpr, tpr, _ = roc_curve(r['y'], r['s'])
            ax.plot(fpr, tpr, color=C[i], label=f"{tag} (AUC={r['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=.4, label='random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name}: ROC curves'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
    figs['roc_curves'] = fig

    # Precision-Recall
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            prec, rec, _ = precision_recall_curve(r['y'], r['s'])
            ax.plot(rec, prec, color=C[i], label=f"{tag} (AP={r['pr_auc']:.3f})")
    ax.axhline(0.5, ls='--', color='k', alpha=.4, label='random (balanced)')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{model_name}: Precision-Recall curves'); ax.legend(loc='lower left'); ax.grid(alpha=.3)
    figs['pr_curves'] = fig

    # Recall@1 vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'recall@1_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='o', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('Recall@1')
    ax.set_title(f'{model_name}: Retrieval difficulty (R@1 vs pool)')
    ax.legend(); ax.grid(alpha=.3)
    figs['recall1_vs_pool'] = fig

    # MRR vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'mrr_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='s', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('MRR')
    ax.set_title(f'{model_name}: MRR vs pool'); ax.legend(); ax.grid(alpha=.3)
    figs['mrr_vs_pool'] = fig

    # Per-axis R@1 grouped bars
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(axes)); w = 0.8 / max(len(tags), 1)
    for i, (tag, r) in enumerate(results.items()):
        p = max(_pools_in(r, pools)); ys = [r.get(f'{a}_recall@1_pool{p}', 0) for a in axes]
        ax.bar(x + i*w, ys, w, color=C[i], label=f'{tag} (pool{p})')
    ax.set_xticks(x + w*(len(tags)-1)/2)
    ax.set_xticklabels([AXIS_LABELS.get(a, a) for a in axes])
    ax.set_ylabel('Recall@1'); ax.set_title(f'{model_name}: Per-axis difficulty (XO > XA > XM)')
    ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['per_axis_bars'] = fig

    # Recall@k by axis
    main = tags[0]; r = results[main]
    p = 100 if f'XO_recall@1_pool100' in r else max(_pools_in(r, pools))
    fig, ax = plt.subplots(figsize=(6, 4))
    for a in axes:
        ys = [r.get(f'{a}_recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=AXIS_LABELS.get(a, a))
    ax.set_xlabel('k'); ax.set_ylabel(f'Recall@k @ pool{p}'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k by axis ({main})'); ax.legend(); ax.grid(alpha=.3)
    figs['recallk_by_axis'] = fig

    # Heatmap: axis x pool
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_recall@1_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis'); ax.set_title(f'{model_name}: R@1 heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['heatmap_axis_pool'] = fig

    # ROC-AUC & PR-AUC bars
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(tags)); w = 0.35
    ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w, label='ROC-AUC')
    ax.bar(x + w/2, [results[t]['pr_auc'] for t in tags], w, label='PR-AUC')
    ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0.5, 1.0)
    ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC & PR-AUC'); ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['auc_prauc_bars'] = fig

    # Score distribution (pos vs neg)
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(s[y == 1], bins=40, alpha=.6, label='same function (pos)', color='green', density=True)
        ax.hist(s[y == 0], bins=40, alpha=.6, label='different (neg)', color='red', density=True)
        ax.set_xlabel('similarity score'); ax.set_ylabel('density')
        ax.set_title(f'{model_name}: Score separation ({main})'); ax.legend(); ax.grid(alpha=.3)
        figs['score_distribution'] = fig


    # Recall@k coverage curve (per pool)
    main = tags[0]; r = results[main]
    fig, ax = plt.subplots(figsize=(6, 4))
    for p in _pools_in(r, pools):
        ys = [r.get(f'recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=f'pool{p}')
    ax.set_xlabel('k'); ax.set_ylabel('Recall@k'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k coverage ({main})')
    ax.legend(); ax.grid(alpha=.3)
    figs['recallk_coverage'] = fig


    # AUC vs retrieval gap
    for k in ks:
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(tags)); w = 0.35
        ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w,
               label='ROC-AUC', color='steelblue')
        # Recall@k at the largest pool available for each eval set
        rk_key = lambda t: f'recall@{k}_pool{max(_pools_in(results[t], pools))}'
        ax.bar(x + w/2, [results[t].get(rk_key(t), 0) for t in tags], w,
               label=f'R@{k} (largest pool)', color='coral')
        ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0, 1)
        ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC vs R@{k} gap')
        ax.legend(); ax.grid(alpha=.3, axis='y')
        figs[f'auc_retrieval_gap_k{k}'] = fig


    # MRR heatmap (axis x pool)
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_mrr_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='magma', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis')
    ax.set_title(f'{model_name}: MRR heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['mrr_heatmap'] = fig


    # Score CDF: cumulative distribution of pos vs neg similarity scores.
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        for lab, mask, col in [('same function (pos)', y == 1, 'green'),
                               ('different (neg)', y == 0, 'red')]:
            v = np.sort(s[mask]); cdf = np.arange(1, len(v)+1) / len(v)
            ax.plot(v, cdf, color=col, label=lab)
        ax.set_xlabel('similarity score'); ax.set_ylabel('cumulative fraction')
        ax.set_title(f'{model_name}: Score CDF ({main})')
        ax.legend(); ax.grid(alpha=.3)
        figs['score_cdf'] = fig

    return figs

### Printing

In [167]:
def print_comparison(results):
    metrics = [('ROC-AUC','roc_auc'), ('R@1@1000','recall@1_pool1000'),
               ('MRR@1000','mrr_pool1000'), ('XO@1@1k','XO_recall@1_pool1000'),
               ('XA@1@1k','XA_recall@1_pool1000'), ('XM@1@1k','XM_recall@1_pool1000')]
    tags = list(results.keys())
    print(f"\n{'metric':>12} | " + " | ".join(f"{t:>14}" for t in tags))
    print("-" * (14 + 3 + len(tags)*17))
    for label, key in metrics:
        row = f"{label:>12} | "
        row += " | ".join(f"{results[t].get(key, float('nan')):>14.4f}" for t in tags)
        print(row)

### Wandb Logging

In [168]:
def _cast(v):
    return float(v) if isinstance(v, (np.floating, np.integer, np.bool_)) else v

def log(run, results, figs, table_name='metrics'):
    # log metrics
    for tag, res in results.items():
        payload = {f'{tag}/{k}': _cast(v) for k, v in res.items()}
        run.log(payload)
        for k, v in payload.items():
            run.summary[k] = v

    table = wandb.Table(columns=['eval_set', 'metric', 'value'])
    for tag, res in results.items():
        for k, v in res.items():
            if k in ('s', 'y'): continue
            table.add_data(tag, k, _cast(v))
    run.log({table_name: table})

    # log images
    run.log({f"plots/{n}": wandb.Image(f) for n, f in figs.items()})

# Model

In [169]:
class FeatureAggregator():
  def embed_batch(self, funcs):
    out = []
    for r in funcs:
      X = r['X']
      out.append(np.concatenate([X.mean(0), X.sum(0), X.max(0), X.std(0), [len(X)]]))
    E = np.asarray(out, dtype=np.float64)
    return (E - E.mean(0)) / (E.std(0) + 1e-8)

# Similarity Functions

### Cos similarity

In [170]:
def cos_sim(E, query_index, cand_index):
  En = _normalize(E)
  return np.dot(En[query_index], En[cand_index].T)

### L2 similarity

In [171]:
def l2_sim(E, query_index, cand_index):
  diff = E[query_index] - E[cand_index]
  return -np.linalg.norm(diff, axis=-1)

 # Read Dataset

In [172]:
def make_funcs_labels(path, min_blocks, n_feat, ratios):
  funcs, labels, classes = load_dataset(path, min_blocks=min_blocks, n_feat=n_feat)
  train_mask, val_mask, test_mask = make_split(labels, ratios=ratios)

  train_funcs = [f for f, m in zip(funcs, train_mask) if m]
  val_funcs   = [f for f, m in zip(funcs, val_mask) if m]
  test_funcs  = [f for f, m in zip(funcs, test_mask) if m]

  train_labels = labels[train_mask]
  val_labels   = labels[val_mask]
  test_labels  = labels[test_mask]

  return train_funcs, val_funcs, test_funcs, train_labels, val_labels, test_labels

In [219]:
min_blocks = 15; n_feat = 21

_, _, openssl_test_funcs, _, _, openssl_test_labels = make_funcs_labels(f'{PATH}/data_acfg/openssl_1_0_1f_acfg', min_blocks=min_blocks, n_feat=n_feat, ratios=(.8, .1, .1))
_, _, zlib_test_funcs,    _, _, zlib_test_labels    = make_funcs_labels(f'{PATH}/data_acfg/zlib_acfg',           min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))
_, _, sqlite_test_funcs,  _, _, sqlite_test_labels  = make_funcs_labels(f'{PATH}/data_acfg/sqlite3_acfg',        min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))

[load] files=12 funcs=18120 classes=2123 dropped(<15 blocks)=52848 n_feat=21
[split] train=14561 val=1747 test=1812
[load] files=12 funcs=1636 classes=228 dropped(<15 blocks)=2398 n_feat=21
[split] train=0 val=0 test=1636
[load] files=12 funcs=9430 classes=1713 dropped(<15 blocks)=16207 n_feat=21
[split] train=0 val=0 test=9430


# Train Model

In [220]:
# Training not needed for this model
model = FeatureAggregator()

# Test model

In [221]:
score_fn=cos_sim; pool_sizes=(10, 100, 1000); n_pairs=8000; n_queries=3000; ks=(1, 5, 10); per_axis=True

datasets = {
    'openssl_1_0_1f': (openssl_test_funcs, openssl_test_labels),
    'zlib-1.3.1':     (zlib_test_funcs,    zlib_test_labels),
    'sqlite3':        (sqlite_test_funcs,  sqlite_test_labels),
}

run = wandb.init(
    project='binsim_baseline_feature_aggregator',
    name=f'feature_aggregator__sim_fn_{score_fn.__name__}__n_feat_{n_feat}__min_blocks_{min_blocks}',
    config = {
        'model': 'feature_aggregate_baseline',
        'sim_fn': score_fn.__name__,
        'n_feat': n_feat,
        'min_blocks': min_blocks,
        'pool_sizes': pool_sizes,
        'n_pairs': n_pairs,
        'n_queries': n_queries,
        'ks': ks,
        'per_axis': per_axis,
        'seed': 1337,
        'arches': ['x86', 'arm32', 'mips32'],
        'opts': ['O0', 'O1', 'O2', 'O3'],
        'openssl_1_0_1f_split_ratios': [0.8, 0.1, 0.1],
        'train_lib': 'openssl-1.0.1f',
        'eval_sets': ['openssl_1_0_1f', 'zlib-1.3.1', 'sqlite3'],
    }
)

results = {}

for tag, (test, labels) in datasets.items():
  results[tag] = evaluate(model, score_fn, test, labels, pool_sizes=pool_sizes, n_pairs=n_pairs, n_queries=n_queries, ks=ks, per_axis=per_axis)

print_comparison(results)

figs = make_plots(results, axes=['XA', 'XO', 'XM'], pools=pool_sizes, ks=ks, model_name='feature_aggregator')
log(run, results, figs)
run.finish()


      metric | openssl_1_0_1f |     zlib-1.3.1 |        sqlite3
--------------------------------------------------------------------
     ROC-AUC |         0.8278 |         0.8687 |         0.8150
    R@1@1000 |         0.1547 |         0.1480 |         0.1130
    MRR@1000 |         0.1952 |         0.1971 |         0.1572
     XO@1@1k |         0.4935 |         0.4915 |         0.4165
     XA@1@1k |         0.0744 |         0.0807 |         0.0580
     XM@1@1k |         0.0283 |         0.0425 |         0.0142


openssl_1_0_1f/XA_mrr_pool10,▁
openssl_1_0_1f/XA_mrr_pool100,▁
openssl_1_0_1f/XA_mrr_pool1000,▁
openssl_1_0_1f/XA_recall@10_pool10,▁
openssl_1_0_1f/XA_recall@10_pool100,▁
openssl_1_0_1f/XA_recall@10_pool1000,▁
openssl_1_0_1f/XA_recall@1_pool10,▁
openssl_1_0_1f/XA_recall@1_pool100,▁
openssl_1_0_1f/XA_recall@1_pool1000,▁
openssl_1_0_1f/XA_recall@5_pool10,▁
+140,...
